# BigQuery Optimization Demo

Interactive demonstration of BigQuery optimization techniques:
- Partition pruning
- Clustering effectiveness
- Materialized views
- Query plan analysis
- Cost estimation

**Run this notebook to see real optimization patterns in action.**

In [ ]:
# Setup
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Initialize BigQuery client
client = bigquery.Client()

# Configuration
PROJECT_ID = 'your-project-id'
DATASET_ID = 'analytics_demo'
TABLE_ID = 'orders_optimized'

print(f"BigQuery client initialized for project: {PROJECT_ID}")

## 1. Create Optimized Table (Partitioned + Clustered)

In [ ]:
# Create optimized table with partitioning and clustering
create_table_sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
PARTITION BY DATE(order_timestamp)
CLUSTER BY customer_id, region, status
OPTIONS (
  description = "Orders table partitioned by date, clustered by customer/region/status",
  require_partition_filter = TRUE
) AS
SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.orders_raw`;
"""

print("Creating optimized table...")
job = client.query(create_table_sql)
job.result()
print("Table created successfully!")

## 2. Compare Query Performance: Before vs After Optimization

In [ ]:
# Query 1: Full scan on raw table (BAD - no partition pruning)
query_bad = f"""
SELECT customer_id, COUNT(*) as order_count, SUM(amount) as total_revenue
FROM `{PROJECT_ID}.{DATASET_ID}.orders_raw`
WHERE order_timestamp >= '2024-01-01'
GROUP BY customer_id
ORDER BY total_revenue DESC
LIMIT 100
"""

# Query 2: Partition pruning + clustering on optimized table (GOOD)
query_good = f"""
SELECT customer_id, COUNT(*) as order_count, SUM(amount) as total_revenue
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
WHERE DATE(order_timestamp) >= '2024-01-01'
GROUP BY customer_id
ORDER BY total_revenue DESC
LIMIT 100
"""

print("Running BAD query (full scan)...")
job_bad = client.query(query_bad)
result_bad = job_bad.result()

print("Running GOOD query (partition pruning + clustering)...")
job_good = client.query(query_good)
result_good = job_good.result()

# Compare metrics
print(f"\n=== PERFORMANCE COMPARISON ===")
print(f"BAD Query:")
print(f"  Bytes billed: {job_bad.total_bytes_billed / 1e9:.2f} GB")
print(f"  Slot ms: {job_bad.total_slot_ms:,}")
print(f"  Estimated cost: ${job_bad.total_bytes_billed / 1e12 * 6.25:.4f}")
print(f"  Execution time: {job_bad.ended - job_bad.started}")

print(f"\nGOOD Query:")
print(f"  Bytes billed: {job_good.total_bytes_billed / 1e9:.2f} GB")
print(f"  Slot ms: {job_good.total_slot_ms:,}")
print(f"  Estimated cost: ${job_good.total_bytes_billed / 1e12 * 6.25:.4f}")
print(f"  Execution time: {job_good.ended - job_good.started}")

improvement = (1 - job_good.total_bytes_billed / job_bad.total_bytes_billed) * 100
print(f"\n🎯 IMPROVEMENT: {improvement:.1f}% reduction in bytes scanned")

## 3. Analyze Query Plan with EXPLAIN

In [ ]:
# Get query execution plan
explain_sql = f"EXPLAIN {query_good}"
job_explain = client.query(explain_sql)
plan = job_explain.result()

print("=== QUERY EXECUTION PLAN ===")
for row in plan:
    print(f"  {row}")

## 4. Materialized View Demo

In [ ]:
# Create materialized view for daily customer aggregates
mv_sql = f"""
CREATE MATERIALIZED VIEW `{PROJECT_ID}.{DATASET_ID}.mv_daily_customer_revenue`
PARTITION BY event_date
CLUSTER BY customer_id, region
OPTIONS (
  enable_refresh = TRUE,
  refresh_interval_minutes = 60
) AS
SELECT 
  DATE(order_timestamp) as event_date,
  customer_id,
  region,
  COUNT(*) as order_count,
  SUM(amount) as revenue
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
GROUP BY 1, 2, 3;
"""

print("Creating materialized view...")
job_mv = client.query(mv_sql)
job_mv.result()
print("Materialized view created!")

In [ ]:
# Query that automatically uses the materialized view
mv_query = f"""
SELECT customer_id, SUM(revenue) as total_revenue
FROM `{PROJECT_ID}.{DATASET_ID}.mv_daily_customer_revenue`
WHERE event_date >= '2024-01-01'
GROUP BY customer_id
ORDER BY total_revenue DESC
LIMIT 10
"""

job_mv_query = client.query(mv_query)
result_mv = job_mv_query.result()

print(f"MV Query bytes billed: {job_mv_query.total_bytes_billed / 1e6:.2f} MB")
print(f"MV Query slot ms: {job_mv_query.total_slot_ms:,}")

for row in result_mv:
    print(f"  {row.customer_id}: ${row.total_revenue:,.2f}")

## 5. Slot Utilization Analysis

In [ ]:
# Analyze slot usage for recent queries
slot_analysis_sql = f"""
SELECT 
  job_id,
  user_email,
  query,
  total_slot_ms / 1000 / 3600 as slot_hours,
  bytes_billed / 1e12 as tb_scanned,
  bytes_billed / 1e12 * 6.25 as estimated_cost_usd,
  creation_time
FROM `region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE 
  creation_time > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
  AND job_type = 'QUERY'
  AND state = 'DONE'
  AND project_id = '{PROJECT_ID}'
ORDER BY bytes_billed DESC
LIMIT 20
"""

job_slots = client.query(slot_analysis_sql)
df_slots = job_slots.to_dataframe()

print(df_slots[['job_id', 'user_email', 'slot_hours', 'tb_scanned', 'estimated_cost_usd']].to_string())

## 6. Partition & Clustering Effectiveness Visualization

In [ ]:
# Get partition and clustering metadata
metadata_sql = f"""
SELECT 
  partition_id,
  total_rows,
  total_logical_bytes,
  total_physical_bytes,
  last_modified_time
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}$__PARTITIONS_SUMMARY__`
WHERE partition_id != '__NULL__'
ORDER BY partition_id DESC
LIMIT 30
"""

job_meta = client.query(metadata_sql)
df_meta = job_meta.to_dataframe()

if not df_meta.empty:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Rows per partition
    axes[0, 0].bar(range(len(df_meta)), df_meta['total_rows'])
    axes[0, 0].set_title('Rows per Partition')
    axes[0, 0].set_xlabel('Partition (recent first)')
    axes[0, 0].set_ylabel('Row Count')
    
    # Logical bytes per partition
    axes[0, 1].bar(range(len(df_meta)), df_meta['total_logical_bytes'] / 1e9)
    axes[0, 1].set_title('Logical Bytes per Partition (GB)')
    axes[0, 1].set_xlabel('Partition (recent first)')
    axes[0, 1].set_ylabel('GB')
    
    # Physical bytes (compressed)
    axes[1, 0].bar(range(len(df_meta)), df_meta['total_physical_bytes'] / 1e9)
    axes[1, 0].set_title('Physical Bytes per Partition (GB)')
    axes[1, 0].set_xlabel('Partition (recent first)')
    axes[1, 0].set_ylabel('GB')
    
    # Compression ratio
    compression_ratio = df_meta['total_logical_bytes'] / df_meta['total_physical_bytes']
    axes[1, 1].bar(range(len(df_meta)), compression_ratio)
    axes[1, 1].set_title('Compression Ratio')
    axes[1, 1].set_xlabel('Partition (recent first)')
    axes[1, 1].set_ylabel('Ratio')
    axes[1, 1].axhline(y=compression_ratio.mean(), color='r', linestyle='--', label=f'Mean: {compression_ratio.mean():.1f}x')
    axes[1, 1].legend()
    
    plt.tight_layout()
    plt.show()
    
    print(df_meta[['partition_id', 'total_rows', 'total_logical_bytes', 'total_physical_bytes']].to_string())
else:
    print("No partition metadata available")

## 7. Cost Optimization Recommendations

In [ ]:
# Cost optimization analysis
cost_sql = f"""
SELECT 
  DATE(creation_time) as query_date,
  COUNT(*) as query_count,
  SUM(total_slot_ms) / 1000 / 3600 as total_slot_hours,
  SUM(bytes_billed) / 1e12 as tb_scanned,
  SUM(bytes_billed) / 1e12 * 6.25 as on_demand_cost_usd,
  AVG(total_slot_ms) / 1000 as avg_slot_seconds
FROM `region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE 
  creation_time > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
  AND job_type = 'QUERY'
  AND state = 'DONE'
  AND project_id = '{PROJECT_ID}'
GROUP BY query_date
ORDER BY query_date DESC
"""

job_cost = client.query(cost_sql)
df_cost = job_cost.to_dataframe()

print("=== 30-DAY COST TREND ===")
print(df_cost.to_string())

# Plot cost trend
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(df_cost['query_date'], df_cost['tb_scanned'], 'o-')
axes[0].set_title('TB Scanned per Day')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('TB')
axes[0].tick_params(axis='x', rotation=45)

axes[1].plot(df_cost['query_date'], df_cost['on_demand_cost_usd'], 'o-', color='orange')
axes[1].set_title('Estimated On-Demand Cost per Day')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('USD')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f"\nTotal 30-day cost: ${df_cost['on_demand_cost_usd'].sum():.2f}")
print(f"Average daily cost: ${df_cost['on_demand_cost_usd'].mean():.2f}")
print(f"Potential savings with 3-yr commitment (35%): ${df_cost['on_demand_cost_usd'].sum() * 0.35:.2f}")

## 8. Optimization Checklist Summary

In [ ]:
# Print the optimization checklist
checklist = """
BIGQUERY OPTIMIZATION CHECKLIST:

☐ 1. PARTITION PRUNING
   WHERE event_date >= '2024-01-01'          ✅ GOOD
   WHERE DATE(event_ts) >= '2024-01-01'      ❌ BAD - function on column
   WHERE event_date = CURRENT_DATE()         ❌ BAD - non-deterministic

☐ 2. CLUSTERING (Max 4 columns, high cardinality first)
   CLUSTER BY customer_id, region, product_category, channel
   → Enables block pruning + sort optimization

☐ 3. COLUMN PRUNING
   SELECT col1, col2 NOT SELECT *

☐ 4. MATERIALIZED VIEWS
   Pre-aggregate frequent GROUP BYs (auto-refresh)

☐ 5. BI ENGINE
   Reserve 50-100GB for sub-second dashboard queries

☐ 6. SLOT ESTIMATION
   Check "slot_ms" in INFORMATION_SCHEMA.JOBS_BY_PROJECT

☐ 7. QUERY PLAN
   Look for "Broadcast" joins → consider clustering both sides

☐ 8. CACHING
   Deterministic UDFs, no NOW()/RAND() → 100% cache hits

☐ 9. RESERVATIONS
   Baseline + Flex for burst, Edition commitment for 30-50% discount

☐ 10. SEARCH OPTIMIZATION
   For point lookups on high-cardinality columns (user_id, session_id)
"""

print(checklist)